In [1]:
import sys
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import math, random
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.tensorboard import SummaryWriter

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, confusion_matrix

import matplotlib.pyplot as plt
from piqa import SSIM

from torchfuzzy import FuzzyBellLayer, DefuzzyNWLayer
from torchinfo import summary


## Params

In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

# ---------------- Hyperparams --------------------
EXCLUDED_DIGIT   = 4        # аномальный класс, НЕ участвует в обучении
LATENT_DIM       = 16
NUM_CLUSTERS     = 18
NUM_STYLE        = 16
IMG_SIZE         = 28

BATCH_SIZE       = 512
EPOCHS           = 120
LR_G             = 2e-4
LR_ENC           = 2e-4
LR_FUZZY         = 2e-5        
BETA1, BETA2     = 0.5, 0.999

# ---- Loss weights (ручной tune) ------------------
W_D_ADV          = 1.0
W_D_FAKE_FZ      = 0.5
W_D_REAL_FZ      = 0.3
W_D_CLUSTER_USE  = 0.8
W_D_PRETANH_L2   = 1e-3

W_G_ADV          = 1.0
ALPHA_SSIM       = 1.0
BETA_L1          = 1.0
W_G_RECON        = 2.0

# ---- Misc ----------------------------------------
GRAD_CLIP        = 5.0
DEAD_EPS         = -(1.0/NUM_CLUSTERS)*np.log(1.0/NUM_CLUSTERS)/5
DEAD_T_EPOCHS    = 4
STYLE_CENTER_SPREAD = 0.3   # initial style centers near 0 with spread
STYLE_SCALE      = 0.4       # diag(A) of style fuzzy terms
CLUSTER_SCALE    = 10       # diag(A) of cluster fuzzy terms (narrower bells on cube)
EPS              = 1e-8

VIZ_EVERY        = 5
CKPT_DIR         = Path('./ckpts');  CKPT_DIR.mkdir(exist_ok=True, parents=True)
LOG_DIR          = Path('./runs') / f'fuzzy_gan_{datetime.now().strftime("%Y%m%d-%H%M%S")}'
writer           = SummaryWriter(LOG_DIR)
print("tb logdir:", LOG_DIR)
print(f"DEAD_EPS={DEAD_EPS}")


device: cuda
tb logdir: runs\fuzzy_gan_20260503-221135
DEAD_EPS=0.03211524175440183


## Dataset

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),              # [0,1]
    transforms.Normalize((0.5,), (0.5,))  # -> [-1,1] под tanh генератора
])

train_full = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_full  = datasets.MNIST('./data', train=False, download=True, transform=transform)

def split_by_digit(ds, excluded):
    labels = ds.targets.numpy() if hasattr(ds, 'targets') else np.array([y for _,y in ds])
    normal_idx   = np.where(labels != excluded)[0]
    anomaly_idx  = np.where(labels == excluded)[0]
    return Subset(ds, normal_idx.tolist()), Subset(ds, anomaly_idx.tolist())

train_normal,  train_anom   = split_by_digit(train_full, EXCLUDED_DIGIT)
test_normal,   test_anom    = split_by_digit(test_full,  EXCLUDED_DIGIT)

train_loader = DataLoader(train_normal, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, drop_last=True, pin_memory=True)
test_loader  = DataLoader(test_full,    batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"train normal: {len(train_normal)}  |  test normal: {len(test_normal)}  |  test anomaly: {len(test_anom)}")


train normal: 54158  |  test normal: 9018  |  test anomaly: 982


## Model

In [4]:
class Encoder(nn.Module):
    """ CNN -> pre_tanh -> tanh(z). Возвращает (z, pre_tanh). """
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1),  
            nn.LeakyReLU(0.2, True),   # 14
            nn.Conv2d(32, 64, 4, 2, 1), 
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, True),                                # 7
            nn.Conv2d(64,128, 3, 2, 1), 
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True),                                # 4
            nn.Flatten(),
        )
        self.head = nn.Linear(128*4*4, latent_dim)

    def forward(self, x):
        h = self.net(x)
        pre = self.head(h)
        z   = torch.tanh(pre)
        return z, pre


class Generator(nn.Module):
    """ (cluster_vec, style_vec) -> image in [-1,1]. """
    def __init__(self, num_clusters=NUM_CLUSTERS, num_style=NUM_STYLE):
        super().__init__()
        in_dim = num_clusters + num_style
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 256*4*4),
            nn.BatchNorm1d(256*4*4), 
            nn.ReLU(True),
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 3, 2, 1, output_padding=0),  # 7
            nn.BatchNorm2d(128), 
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), 
            nn.BatchNorm2d(64), # 14
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1),                       # 28
            nn.Tanh(),
        )

    def forward(self, c, s):
        z = torch.cat([c, s], dim=1)
        h = self.fc(z).view(-1, 256, 4, 4)
        return self.deconv(h)


def build_fuzzy_cluster(latent_dim, num_clusters):
    """ Центры на поверхности куба [-1,1]^D. """
    # Случайно выбираем для каждого центра случайную вершину+поверхность куба
    centers = torch.empty(num_clusters, latent_dim).uniform_(-1, 1)
    # «приклеиваем» по одной случайной координате к ±1, чтобы точка была на поверхности
    rows = torch.arange(num_clusters)
    fixed_dim = torch.randint(0, latent_dim, (num_clusters,))
    signs     = torch.where(torch.rand(num_clusters) > 0.5, 1.0, -1.0)
    centers[rows, fixed_dim] = signs
    scales = torch.full_like(centers, CLUSTER_SCALE)
    return FuzzyBellLayer(centers, scales, b_parametrization="softplus")


def build_fuzzy_style(latent_dim, num_style):
    """ Центры стиля около нуля со спредом ~STYLE_CENTER_SPREAD, scales ~0.3-0.5. """
    centers = torch.randn(num_style, latent_dim) * STYLE_CENTER_SPREAD
    scales  = torch.full_like(centers, STYLE_SCALE)
    return FuzzyBellLayer(centers, scales, b_parametrization="softplus")


class FuzzyDiscriminator(nn.Module):
    """
    Собирает всё: Encoder -> FuzzyLayer_cluster -> residual -> FuzzyLayer_style
                 -> T-norm rules -> DefuzzyNWLayer -> scalar firing.
    """
    def __init__(self, latent_dim=LATENT_DIM,
                 num_clusters=NUM_CLUSTERS, num_style=NUM_STYLE):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.fuzzy_c = build_fuzzy_cluster(latent_dim, num_clusters)
        self.fuzzy_s = build_fuzzy_style(latent_dim, num_style)

        num_rules = num_clusters * num_style
        # НАЧАЛЬНЫЕ «заключения» (consequences): маленькие случайные, выход 1-мерный
        init_Z = torch.randn(1, num_rules) * 0.1
        self.defuzzy = DefuzzyNWLayer(init_Z, trainable=True, with_norm=True)

        self.num_clusters = num_clusters
        self.num_style    = num_style

    def fuzzy_forward(self, z):
        """ z -> (fz_cluster, residual, fz_style, rules, firing). """
        fz_c = self.fuzzy_c(z)                                # (B, K_c)

        # residual = z -  centers_c[argsoftmax(fz_c)]
        #   центры ДЕТАЧИМ (не двигаются через градиент по residual),
        #   fz_c сохраняет градиент.
        centers_c = self.fuzzy_c.centers.detach()             # (K_c, D)
        #num  = fz_c.unsqueeze(-1) * centers_c.unsqueeze(0)    # (B, K_c, D)
        #num  = num.sum(dim=1)                                 # (B, D)
        #den  = fz_c.sum(dim=1, keepdim=True).clamp_min(EPS)   # (B, 1)
        #weighted_c = num / den                                # (B, D)
        #residual   = z - weighted_c                           # (B, D)
        
        idx = fz_c.argmax(dim=-1)
        mask = F.one_hot(idx, num_classes=fz_c.size(-1)).float()
        soft = fz_c.softmax(dim=-1)
        ste_mask = mask + (soft - soft.detach())
        target_center = ste_mask @ centers_c
        residual = z - target_center

        fz_s = self.fuzzy_s(residual)                         # (B, K_s)

        # T-norm (произведение): rules[i,j] = fz_c[i] * fz_s[j]
        rules = fz_c.unsqueeze(2) * fz_s.unsqueeze(1)         # (B, K_c, K_s)
        rules = rules.flatten(1)                              # (B, K_c*K_s)

        firing = self.defuzzy(rules).squeeze(-1)              # (B,)
        return dict(fz_c=fz_c, fz_s=fz_s, residual=residual,
                    rules=rules, firing=firing)

    def forward(self, x):
        z, pre = self.encoder(x)
        out = self.fuzzy_forward(z)
        out['z'] = z
        out['pre'] = pre
        return out


## Losses

In [5]:
# piqa.SSIM работает с float в [0,1], multi-channel=1
ssim_metric = SSIM(window_size=7, n_channels=1).to(device)

def hinge_d_loss(d_real, d_fake):
    """ Combined L_real + L_fake """
    return F.relu(1. - d_real).mean() + F.relu(1. + d_fake).mean()

def hinge_g_loss(d_fake_for_g):
    """ Generator хочет поднять свой firing выше −1; классика Hinge-GAN: −E[D(fake)]. """
    return -d_fake_for_g.mean()

def mse(a, b):  return F.mse_loss(a, b)

def l_fake_fz(nz_c, nz_s, fz_c, fz_s):
    # (nz_c - fz_c)^2 + (nz_s - fz_s)^2
    return mse(fz_c, nz_c) + mse(fz_s, nz_s)

def l_real_fz(fz_c):
    m = fz_c.max(dim=1).values                  # (B,)
    s = fz_c.sum(dim=1)                         # (B,)
    return ((1. - m) ** 2).mean() + ((1. - s) ** 2).mean()

def l_cluster_usage(fz_c, eps=EPS):
    """ Хотим максимизировать энтропию использования кластеров по батчу.
        Формула из ТЗ: L = -(-sum p_bar*log p_bar) = -H(p_bar).
        Минимизация L == максимизация энтропии. """
    p_bar = fz_c.mean(dim=0)
    p_bar = p_bar / (p_bar.sum() + eps)
    H = -(p_bar * (p_bar + eps).log()).sum()
    return -H

def l_reconstruct(x_real, x_reg, alpha=ALPHA_SSIM, beta=BETA_L1):
    # SSIM требует [0,1]
    xr = (x_real.clamp(-1, 1) + 1.) / 2.
    xg = (x_reg.clamp(-1, 1) + 1.) / 2.
    ssim_val = ssim_metric(xr, xg)
    l1       = F.l1_loss(x_reg, x_real)
    return alpha * (1. - ssim_val) + beta * l1


## Helpers

In [6]:
def sample_nz(batch_size, num_clusters=NUM_CLUSTERS, num_style=NUM_STYLE,
              soft_max=0.98, soft_other=0.0, style_low=0.0, style_high=1.0, device=device):
    """ nz_cluster: one-hot-ish (одна ~0.98, остальные 0),
        nz_style:   U[0,1] по каждой координате. """
    idx   = torch.randint(0, num_clusters, (batch_size,), device=device)
    nz_c  = torch.full((batch_size, num_clusters), soft_other, device=device)
    nz_c[torch.arange(batch_size, device=device), idx] = soft_max
    nz_s  = torch.empty(batch_size, num_style, device=device).uniform_(style_low, style_high)
    return nz_c, nz_s, idx


@torch.no_grad()
def reinit_dead_clusters(D, last_batch_z, last_batch_fz_c, dead_flags):
    """ Переместить мёртвые кластеры в 'худший' z (по max(fz_cluster)). """
    if not dead_flags.any():
        return 0
    # 'худший' сэмпл: у которого максимальный fz_c наименьший
    worst_score = last_batch_fz_c.max(dim=1).values  # (B,)
    _, order = torch.sort(worst_score)               # возрастание
    n_dead = int(dead_flags.sum().item())
    take = order[:n_dead]
    new_centers = last_batch_z[take].clone()         # (n_dead, D)

    centers = D.fuzzy_c.centers.data
    dead_idx = torch.nonzero(dead_flags, as_tuple=False).view(-1)
    for k, newc in zip(dead_idx, new_centers):
        centers[k] = newc
    return n_dead


def to_numpy(t): return t.detach().cpu().numpy()


def fig_to_image_tensor(fig):
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    return torch.from_numpy(img.copy()).permute(2, 0, 1) / 255.


@torch.no_grad()
def viz_tsne_latent(D, loader, writer, step, max_samples=1500, tag_prefix="tsne"):
    D.eval()
    zs, resids, fzc, labels = [], [], [], []
    cnt = 0
    for x, y in loader:
        x = x.to(device)
        out = D(x)
        zs.append(to_numpy(out['z']))
        resids.append(to_numpy(out['residual']))
        fzc.append(to_numpy(out['fz_c']))
        labels.append(y.numpy())
        cnt += x.size(0)
        if cnt >= max_samples: break
    z      = np.concatenate(zs)[:max_samples]
    resid  = np.concatenate(resids)[:max_samples]
    fzc    = np.concatenate(fzc)[:max_samples]
    labels = np.concatenate(labels)[:max_samples]
    c_assign = fzc.argmax(axis=1)

    c_centers = to_numpy(D.fuzzy_c.centers)                       # (K_c, D)
    s_centers = to_numpy(D.fuzzy_s.centers)                       # (K_s, D)

    # --- TSNE latent + cluster centers ---
    combo = np.concatenate([z, c_centers], axis=0)
    tsne  = TSNE(n_components=2, perplexity=30, init='pca', random_state=SEED)
    emb   = tsne.fit_transform(combo)
    emb_z, emb_c = emb[:len(z)], emb[len(z):]

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    sc0 = ax[0].scatter(emb_z[:,0], emb_z[:,1], c=c_assign, cmap='tab10', s=6, alpha=.7)
    ax[0].scatter(emb_c[:,0], emb_c[:,1], c='black', marker='X', s=120, edgecolors='red')
    for i, p in enumerate(emb_c): 
        ax[0].annotate(str(i), (p[0], p[1]), color='white', fontsize=9)
    ax[0].set_title("TSNE(z) — цвет = argmax(fz_c) + центры кластеров")
    plt.colorbar(sc0, ax=ax[0])

    # --- TSNE residual + style centers ---
    combo2 = np.concatenate([resid, s_centers], axis=0)
    emb2   = TSNE(n_components=2, perplexity=30, init='pca', random_state=SEED).fit_transform(combo2)
    emb_r, emb_s = emb2[:len(resid)], emb2[len(resid):]
    sc1 = ax[1].scatter(emb_r[:,0], emb_r[:,1], c=labels, cmap='tab10', s=6, alpha=.7)
    ax[1].scatter(emb_s[:,0], emb_s[:,1], c='black', marker='X', s=120, edgecolors='red')
    for i, p in enumerate(emb_s): 
        ax[1].annotate(str(i), (p[0], p[1]), color='white', fontsize=8)
    ax[1].set_title("TSNE(residual) — цвет = true label + стилевые центры")
    plt.colorbar(sc1, ax=ax[1])
    writer.add_image(f"{tag_prefix}/latent_and_residual", fig_to_image_tensor(fig), step)


@torch.no_grad()
def viz_prototypes(G, writer, step, num_clusters=NUM_CLUSTERS, num_style=NUM_STYLE, soft_max=0.98):
    G.eval()
    nz_c = torch.full((num_clusters, num_clusters), 0., device=device)
    for i in range(num_clusters):
        nz_c[i, i] = soft_max
    nz_s = torch.zeros(num_clusters, num_style, device=device)    # нулевые стили
    imgs = G(nz_c, nz_s)
    grid = make_grid((imgs.clamp(-1,1)+1)/2, nrow=num_clusters)
    writer.add_image("viz/prototypes_zero_style", grid, step)


@torch.no_grad()
def viz_centroid_tracking_pca(centroid_history_c, centroid_history_s, writer, step):
    """ centroid_history_*: list of (epoch, np.ndarray(K, D)) """
    for name, hist in [("cluster", centroid_history_c), ("style", centroid_history_s)]:
        if len(hist) < 2: continue
        eps_list, arrs = zip(*hist)
        allpts = np.concatenate(arrs, axis=0)                      # (len*K, D)
        pca = PCA(n_components=2).fit(allpts)
        fig, ax = plt.subplots(figsize=(7, 6))
        K = arrs[0].shape[0]
        colors = plt.cm.tab20(np.linspace(0, 1, K))
        for k in range(K):
            traj = np.stack([pca.transform(a[k:k+1])[0] for a in arrs], axis=0)  # (T,2)
            ax.plot(traj[:,0], traj[:,1], '-o', color=colors[k], ms=3, label=f'{k}')
            ax.scatter(traj[-1,0], traj[-1,1], color=colors[k], edgecolors='black', s=80)
        ax.set_title(f"{name} centroids trajectories (PCA)")
        if K <= 12:
            ax.legend(fontsize=7, loc='best')
        writer.add_image(f"centroids/{name}_pca", fig_to_image_tensor(fig), step)


## Training

In [7]:
D = FuzzyDiscriminator(LATENT_DIM, NUM_CLUSTERS, NUM_STYLE).to(device)
G = Generator(NUM_CLUSTERS, NUM_STYLE).to(device)

In [8]:
summary(D, input_size=(BATCH_SIZE, 1, IMG_SIZE, IMG_SIZE))

Layer (type:depth-idx)                   Output Shape              Param #
FuzzyDiscriminator                       [512, 16]                 --
├─Encoder: 1-1                           [512, 16]                 --
│    └─Sequential: 2-1                   [512, 2048]               --
│    │    └─Conv2d: 3-1                  [512, 32, 14, 14]         544
│    │    └─LeakyReLU: 3-2               [512, 32, 14, 14]         --
│    │    └─Conv2d: 3-3                  [512, 64, 7, 7]           32,832
│    │    └─BatchNorm2d: 3-4             [512, 64, 7, 7]           128
│    │    └─LeakyReLU: 3-5               [512, 64, 7, 7]           --
│    │    └─Conv2d: 3-6                  [512, 128, 4, 4]          73,856
│    │    └─BatchNorm2d: 3-7             [512, 128, 4, 4]          256
│    │    └─LeakyReLU: 3-8               [512, 128, 4, 4]          --
│    │    └─Flatten: 3-9                 [512, 2048]               --
│    └─Linear: 2-2                       [512, 16]                 32,784


In [9]:
summary(G, input_size=[(BATCH_SIZE, NUM_CLUSTERS), (BATCH_SIZE, NUM_STYLE)])

Layer (type:depth-idx)                   Output Shape              Param #
Generator                                [512, 1, 28, 28]          --
├─Sequential: 1-1                        [512, 4096]               --
│    └─Linear: 2-1                       [512, 4096]               143,360
│    └─BatchNorm1d: 2-2                  [512, 4096]               8,192
│    └─ReLU: 2-3                         [512, 4096]               --
├─Sequential: 1-2                        [512, 1, 28, 28]          --
│    └─ConvTranspose2d: 2-4              [512, 128, 7, 7]          295,040
│    └─BatchNorm2d: 2-5                  [512, 128, 7, 7]          256
│    └─ReLU: 2-6                         [512, 128, 7, 7]          --
│    └─ConvTranspose2d: 2-7              [512, 64, 14, 14]         131,136
│    └─BatchNorm2d: 2-8                  [512, 64, 14, 14]         128
│    └─ReLU: 2-9                         [512, 64, 14, 14]         --
│    └─ConvTranspose2d: 2-10             [512, 1, 28, 28]        

In [ ]:

# Отдельные param groups — отдельный LR для FuzzyLayer / Defuzzy
enc_params   = list(D.encoder.parameters())
fuzzy_params = list(D.fuzzy_c.parameters()) + list(D.fuzzy_s.parameters()) + list(D.defuzzy.parameters())

opt_D = torch.optim.Adam([
    {'params': enc_params,   'lr': LR_ENC},
    {'params': fuzzy_params, 'lr': LR_FUZZY},
], betas=(BETA1, BETA2))

opt_G = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(BETA1, BETA2))

# Для отслеживания мёртвых кластеров
dead_streak = torch.zeros(NUM_CLUSTERS, dtype=torch.long)
centroid_hist_c, centroid_hist_s = [], []

def detailed_log(writer, step, d_metrics, g_metrics):
    for k, v in d_metrics.items(): writer.add_scalar(f'D/{k}', v, step)
    for k, v in g_metrics.items(): writer.add_scalar(f'G/{k}', v, step)

global_step = 0
for epoch in range(1, EPOCHS + 1):
    D.train(); G.train()
    ep_metrics_d = defaultdict(float)
    ep_metrics_g = defaultdict(float)
    ep_pbar = torch.zeros(NUM_CLUSTERS, device=device)
    n_batches = 0
    last_batch_z = None
    last_batch_fzc = None

    for x_real, _ in train_loader:
        x_real = x_real.to(device, non_blocking=True)
        bs = x_real.size(0)

        # ===================== PHASE D =====================
        opt_D.zero_grad(set_to_none=True)

        # --- fake pass ---
        nz_c, nz_s, _ = sample_nz(bs)
        with torch.no_grad():
            x_fake = G(nz_c, nz_s)
        out_f = D(x_fake)
        # --- real pass ---
        out_r = D(x_real)

        last_batch_z   = out_r['z'].detach()
        last_batch_fzc = out_r['fz_c'].detach()

        # Основной Hinge
        L_adv = hinge_d_loss(out_r['firing'], out_f['firing'])

        # Вспомогательные fuzzy-лоссы
        L_ffz = l_fake_fz(nz_c, nz_s, out_f['fz_c'], out_f['fz_s'])
        L_rfz = l_real_fz(out_r['fz_c'])

        # Энтропийная регуляризация по батчу (реальные)
        L_use = l_cluster_usage(out_r['fz_c'])

        # Pre-tanh L2 (от коллапса энкодера в углы)
        L_pre = (out_r['pre'].pow(2).mean() + out_f['pre'].pow(2).mean()) * 0.5

        L_D = (W_D_ADV         * L_adv +
               W_D_FAKE_FZ     * L_ffz +
               W_D_REAL_FZ     * L_rfz +
               W_D_CLUSTER_USE * L_use +
               W_D_PRETANH_L2  * L_pre)

        L_D.backward()
        # Gradient clipping ("клипинг на бэкпропе D")
        torch.nn.utils.clip_grad_norm_(enc_params + fuzzy_params, GRAD_CLIP)
        opt_D.step()

        ep_metrics_d['adv']        += L_adv.item()
        ep_metrics_d['fake_fz']    += L_ffz.item()
        ep_metrics_d['real_fz']    += L_rfz.item()
        ep_metrics_d['cluster_use']+= L_use.item()
        ep_metrics_d['pretanh_l2'] += L_pre.item()
        ep_metrics_d['total']      += L_D.item()
        ep_metrics_d['D_real_mean']+= out_r['firing'].mean().item()
        ep_metrics_d['D_fake_mean']+= out_f['firing'].mean().item()

        # Накопление p_bar по эпохе для мониторинга мёртвых кластеров
        ep_pbar += out_r['fz_c'].detach().mean(dim=0)

        # ===================== PHASE G =====================
        opt_G.zero_grad(set_to_none=True)

        # ---- adversarial (fake) ----
        nz_c, nz_s, _ = sample_nz(bs)
        x_fake_g = G(nz_c, nz_s)
        out_fg   = D(x_fake_g)  # D через forward, но оптимайзер только для G
        L_g_adv  = hinge_g_loss(out_fg['firing'])

        # ---- reconstruction via cycle: x_real -> E -> fz -> G -> x_regen ----
        with torch.no_grad():
            # получим fz_c, fz_s от дискриминатора (его веса не обновляем здесь)
            z_r, _ = D.encoder(x_real)
        # Чтобы градиент шёл только в G:
        #   fz_c, fz_s — функции z_r; z_r уже detached, поэтому fz тоже как константы.
        out_rg = D.fuzzy_forward(z_r)
        fz_c_d = out_rg['fz_c'].detach()
        fz_s_d = out_rg['fz_s'].detach()
        x_regen = G(fz_c_d, fz_s_d)
        L_g_rec = l_reconstruct(x_real, x_regen, ALPHA_SSIM, BETA_L1)

        L_G = W_G_ADV * L_g_adv + W_G_RECON * L_g_rec
        L_G.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), GRAD_CLIP)
        opt_G.step()

        ep_metrics_g['adv']    += L_g_adv.item()
        ep_metrics_g['recon']  += L_g_rec.item()
        ep_metrics_g['total']  += L_G.item()

        n_batches += 1
        global_step += 1

    # --------- Эпохальные метрики ---------
    for k in ep_metrics_d: 
        ep_metrics_d[k] /= n_batches
    for k in ep_metrics_g: 
        ep_metrics_g[k] /= n_batches
        
    detailed_log(writer, epoch, ep_metrics_d, ep_metrics_g)
    p_bar_ep = (ep_pbar / n_batches).cpu()
    for k in range(NUM_CLUSTERS):
        writer.add_scalar(f'p_bar/cluster_{k}', p_bar_ep[k].item(), epoch)

    print(f"[E{epoch:03d}] "
          f"D_adv={ep_metrics_d['adv']:.3f}  D_ffz={ep_metrics_d['fake_fz']:.3f}  "
          f"D_rfz={ep_metrics_d['real_fz']:.3f}  D_use={ep_metrics_d['cluster_use']:.3f}  "
          f"| G_adv={ep_metrics_g['adv']:.3f}  G_rec={ep_metrics_g['recon']:.3f}  "
          f"| Dr={ep_metrics_d['D_real_mean']:+.2f}  Df={ep_metrics_d['D_fake_mean']:+.2f}")

    # --------- Мёртвые кластеры ---------
    dead_now = (p_bar_ep < DEAD_EPS)
    dead_streak = torch.where(dead_now, dead_streak + 1, torch.zeros_like(dead_streak))
    streak_threshold = DEAD_T_EPOCHS
    if epoch < 10:
        streak_threshold = 2
    to_reinit = dead_streak >= streak_threshold
    
    if to_reinit.any() and last_batch_z is not None:
        n_re = reinit_dead_clusters(D, last_batch_z, last_batch_fzc, to_reinit)
        print(f"  >>> reinit {n_re} cluster centers: {torch.nonzero(to_reinit).view(-1).tolist()}")
        dead_streak[to_reinit] = 0

    # --------- Трекинг центров ---------
    centroid_hist_c.append((epoch, D.fuzzy_c.centers.detach().cpu().numpy().copy()))
    centroid_hist_s.append((epoch, D.fuzzy_s.centers.detach().cpu().numpy().copy()))

    # --------- Визуализации ---------
    if epoch % VIZ_EVERY == 0 or epoch == 1 or epoch == EPOCHS:
        viz_tsne_latent(D, test_loader, writer, epoch)
        viz_prototypes(G, writer, epoch)
        viz_centroid_tracking_pca(centroid_hist_c, centroid_hist_s, writer, epoch)
        torch.save({'D': D.state_dict(), 'G': G.state_dict()},
                   CKPT_DIR / f'ckpt_{epoch:03d}.pt')

writer.flush()
print("Training done.")


[E001] D_adv=1.986  D_ffz=0.137  D_rfz=1.948  D_use=-2.887  | G_adv=0.018  G_rec=1.403  | Dr=-0.00  Df=-0.02
[E002] D_adv=1.979  D_ffz=0.140  D_rfz=1.943  D_use=-2.887  | G_adv=0.022  G_rec=1.253  | Dr=-0.00  Df=-0.02
  >>> reinit 18 cluster centers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
[E003] D_adv=1.996  D_ffz=0.135  D_rfz=1.770  D_use=-2.877  | G_adv=0.005  G_rec=1.301  | Dr=-0.00  Df=-0.00
[E004] D_adv=1.994  D_ffz=0.137  D_rfz=1.651  D_use=-2.880  | G_adv=0.006  G_rec=1.235  | Dr=-0.00  Df=-0.01
  >>> reinit 18 cluster centers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
[E005] D_adv=1.995  D_ffz=0.138  D_rfz=1.281  D_use=-2.852  | G_adv=0.004  G_rec=1.286  | Dr=+0.00  Df=-0.00
[E006] D_adv=1.993  D_ffz=0.138  D_rfz=1.097  D_use=-2.850  | G_adv=0.005  G_rec=1.211  | Dr=+0.00  Df=-0.01
[E007] D_adv=1.991  D_ffz=0.138  D_rfz=0.989  D_use=-2.850  | G_adv=0.006  G_rec=1.195  | Dr=+0.00  Df=-0.01
[E008] D_adv=1.990  D_ffz=0.138  D_rfz=0.955 

## Validation

In [ ]:
@torch.no_grad()
def show_prototypes(G, num_clusters=NUM_CLUSTERS, num_style=NUM_STYLE):
    G.eval()
    nz_c = torch.eye(num_clusters, device=device) * 0.98
    nz_s = torch.zeros(num_clusters, num_style, device=device)
    imgs = G(nz_c, nz_s).clamp(-1,1)
    imgs = (imgs + 1) / 2
    fig, axs = plt.subplots(1, num_clusters, figsize=(num_clusters*1.2, 1.4))
    for k in range(num_clusters):
        axs[k].imshow(imgs[k,0].cpu(), cmap='gray'); axs[k].axis('off')
        axs[k].set_title(f"c={k}", fontsize=9)
    plt.suptitle("Прототипы (style = 0)"); plt.show()

show_prototypes(G)


@torch.no_grad()
def collect_predictions(D, loader):
    D.eval()
    all_fzc, all_fir, all_y = [], [], []
    for x, y in loader:
        x = x.to(device)
        out = D(x)
        all_fzc.append(to_numpy(out['fz_c']))
        all_fir.append(to_numpy(out['firing']))
        all_y.append(y.numpy())
    return np.concatenate(all_fzc), np.concatenate(all_fir), np.concatenate(all_y)

fzc_test, fir_test, y_test = collect_predictions(D, test_loader)
cluster_assign = fzc_test.argmax(axis=1)

# -------- Точность «присвоения меток классам» (unsupervised mapping) --------
# Сопоставим каждому кластеру "свою" истинную метку по мажоритарному классу
# среди НЕаномальных (y != EXCLUDED_DIGIT)
mask_normal = (y_test != EXCLUDED_DIGIT)
ya = y_test[mask_normal]
ca = cluster_assign[mask_normal]

cluster_to_label = {}
for k in range(NUM_CLUSTERS):
    idx = np.where(ca == k)[0]
    if len(idx) == 0:
        cluster_to_label[k] = -1
        continue
    vals, cnts = np.unique(ya[idx], return_counts=True)
    cluster_to_label[k] = int(vals[np.argmax(cnts)])

pred_label = np.array([cluster_to_label[k] for k in ca])
acc = (pred_label == ya).mean()
ari = adjusted_rand_score(ya, ca)
nmi = normalized_mutual_info_score(ya, ca)
print(f"[Normal classes] Purity-based accuracy: {acc*100:.2f}%  |  ARI={ari:.3f}  NMI={nmi:.3f}")
print("cluster -> label :", cluster_to_label)

cm = confusion_matrix(ya, pred_label, labels=sorted(set(ya)))
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm, cmap='Blues')
ax.set_title(f"Confusion matrix (norm. classes), acc={acc*100:.2f}%")
ax.set_xlabel('pred'); ax.set_ylabel('true')
ax.set_xticks(range(len(cm))); ax.set_xticklabels(sorted(set(ya)))
ax.set_yticks(range(len(cm))); ax.set_yticklabels(sorted(set(ya)))
for (i,j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha='center', va='center',
            color='white' if v > cm.max()/2 else 'black', fontsize=8)
plt.colorbar(im); plt.show()


# -------- Выявление аномального класса --------
# Гипотеза: для аномалий (y == EXCLUDED_DIGIT) имеем:
#   - низкий max(fz_cluster)
#   - низкое firing (D < порога)
anomaly_score_a = 1.0 - fzc_test.max(axis=1)          # "далеко от всех кластеров"
anomaly_score_b = -fir_test                           # чем ниже firing — тем "фейковее/дальше"
# Комбинируем z-score
def zscore(x): m,s = x.mean(), x.std()+1e-8; return (x-m)/s
anomaly_score = zscore(anomaly_score_a) + zscore(anomaly_score_b)

is_anom = (y_test == EXCLUDED_DIGIT).astype(int)

# ROC-AUC (ручной)
from sklearn.metrics import roc_auc_score, roc_curve
auc = roc_auc_score(is_anom, anomaly_score)
print(f"Anomaly detection ({EXCLUDED_DIGIT} vs rest) AUC: {auc:.4f}")

fpr, tpr, _ = roc_curve(is_anom, anomaly_score)
plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f'AUC={auc:.3f}')
plt.plot([0,1],[0,1],'--',c='gray'); plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title(f'ROC: class {EXCLUDED_DIGIT} as anomaly'); plt.legend(); plt.show()

# Распределения скоров
fig, ax = plt.subplots(1, 2, figsize=(12,4))
for v, n in [(0, 'normal'), (1, 'anomaly')]:
    ax[0].hist(anomaly_score_a[is_anom==v], bins=50, alpha=.5, label=n, density=True)
    ax[1].hist(fir_test[is_anom==v], bins=50, alpha=.5, label=n, density=True)
ax[0].set_title('1 - max(fz_c)'); ax[0].legend()
ax[1].set_title('D_firing');      ax[1].legend()
plt.show()

# Гистограмма по настоящим классам
avg_anom_by_class = [anomaly_score[y_test==c].mean() for c in range(10)]
plt.figure(figsize=(7,3))
plt.bar(range(10), avg_anom_by_class,
        color=['red' if c==EXCLUDED_DIGIT else 'steelblue' for c in range(10)])
plt.xticks(range(10)); plt.title("Средний anomaly-score по истинному классу")
plt.xlabel('class'); plt.ylabel('score'); plt.show()
